# Genotype–Phenotype Heterogeneity and Heterogeneity-Driven Infertility Management in Non-Classical 21-Hydroxylase Deficiency: A Clinical Analysis and Literature Review Exploration with `mlcroissant`
This notebook demonstrates loading and exploring the FAIR² dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant --quiet

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the dataset URL
croissant_url = "https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json"

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Access metadata (do NOT treat as dict/list)
metadata = dataset.metadata
print("--- Dataset Overview ---")
print(f"Title: {metadata.name}")
print(f"Description: {metadata.description}")
print(f"Authors (@id): {[author['@id'] for author in metadata.author]}")
print(f"Keywords: {', '.join(metadata.keywords)}")
print(f"Date Published: {metadata.datePublished}")
print(f"Identifier: {metadata.identifier}")
print(f"License: {metadata.license}")

## 2. Data Overview
Review available record sets, fields, their `@id` values, and associated columns.

The dataset contains multiple record sets (tables). For each record set, we list its `@id`, name, and the `@id`s for the fields (columns).

In [ ]:
print("--- Record Sets Overview ---")

# List available record sets in the dataset
record_sets = dataset.record_sets
record_set_ids = []

for rs in record_sets:
    print(f"RecordSet name: {rs.name}")
    print(f"  @id: {rs['@id']}")
    record_set_ids.append(rs['@id'])
    print(f"  Description: {rs.description if hasattr(rs, 'description') else ''}")
    # List fields
    print("  Fields:")
    for field in rs.fields:
        print(f"    - {field['@id']} ({field.name})")
    print("-------------------")

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis. Each table is referenced by its record set `@id`, and fields (columns) use their `@id`.

We'll extract all record sets and show their column names and first rows.

In [ ]:
# Extract data from each record set
dataframes = {}

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"\nRecordSet @id: {record_set_id}")
    print(f"Columns (@id): {df.columns.tolist()}")
    print(df.head())

## 4. Exploratory Data Analysis (EDA)
Apply basic processing: filtering records by numeric criteria, normalization, and grouping.

Below, we select numeric fields and demonstrate filtering/normalizing and then try grouping by a relevant field, using `@id` references throughout.

In [ ]:
# Pick a record set with hormone levels (Table 2: Hormone Levels and Imaging Results)
# For demonstration, we select the first record set with numeric fields
rs_with_numeric = None
numeric_field_id = None

for rs in dataset.record_sets:
    # Find numeric field
    for field in rs.fields:
        if hasattr(field, 'dataType') and field.dataType in ['Float', 'Integer', 'Number']:
            rs_with_numeric = rs['@id']
            numeric_field_id = field['@id']
            break
    if rs_with_numeric:
        break

if rs_with_numeric and numeric_field_id:
    df = dataframes[rs_with_numeric]
    print(f"Selected RecordSet @id: {rs_with_numeric}")
    print(f"Selected Numeric Field @id: {numeric_field_id}")

    # Threshold filtering
    threshold = 10
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    print(filtered_df.head())

    # Normalization
    norm_col = f"{numeric_field_id}_normalized"
    filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, norm_col]].head())

    # Try grouping by another field (e.g., phenotype field if present)
    group_field = None
    for field in dataset.record_sets[0].fields:
        if field['@id'] != numeric_field_id:
            group_field = field['@id']
            break

    if group_field and group_field in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean().reset_index()
        print(f"Grouped data by {group_field}: (mean {numeric_field_id})")
        print(grouped_df.head())
else:
    print("No suitable numeric field found for demonstration.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

Below, we plot the distribution of the selected numeric field and, if available, its normalized values. Grouped means are shown if grouping was successful.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if rs_with_numeric and numeric_field_id:
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    sns.histplot(df[numeric_field_id], kde=True, ax=axes[0])
    axes[0].set_title(f"Distribution of {numeric_field_id} (original)")

    if norm_col in df.columns:
        sns.histplot(df[norm_col], kde=True, ax=axes[1])
        axes[1].set_title(f"Distribution of {norm_col} (normalized)")

    plt.tight_layout()
    plt.show()

    # Scatter plot if grouped_df exists
    if 'grouped_df' in locals():
        plt.figure(figsize=(8,5))
        sns.barplot(x=grouped_df[group_field], y=grouped_df[numeric_field_id])
        plt.title(f"Mean {numeric_field_id} grouped by {group_field}")
        plt.xlabel(group_field)
        plt.ylabel(f"Mean {numeric_field_id}")
        plt.show()

## 6. Conclusion
This notebook demonstrated how to:
- Load and inspect the dataset metadata via Croissant schema using `mlcroissant`.
- Explore each record set, referencing all entities by their `@id`.
- Extract the tables with their columns (each referenced as `@id`).
- Apply filtering and normalization to numeric fields and group by key attributes.
- Visualize field distributions and relationships.

**Key observations:**
- The FAIR² dataset covers clinical, hormone, and genetic features of three female patients with non-classical 21-hydroxylase deficiency, described in properly structured tables.
- The schema enables flexible, reproducible access and manipulation of each table using `@id` references, supporting FAIR principles for medical and research datasets.
- Due to limited sample size and cohort specificity, statistical conclusions are not generalizable and should be interpreted as case-level insights.